# TRAIN ONE ABLATION ROW

Run this on each machine with a **different `RUN`**. Machines never communicate;
they only share the same data files.

| machine | set `RUN` to | status |
|---|---|---|
| yours | `baseline` | DONE (bpb 0.4814) |
| machine 2 | `rope` | todo |
| machine 3 | `rope_gqa` | todo |
| machine 4 *(optional)* | `rope_gqa_matched` | todo |

## Files each machine needs (6 files, ~900 MB)
```
train.bin          890 MiB
train.bin.meta     <- exact filename, NOT train_bin.meta
val.bin            9 MiB
val.bin.meta
bpe8192-vocab.json
bpe8192-merges.txt
```
Copy these — do NOT regenerate them. A rebuilt tokenizer produces a different
vocab, which makes bits-per-byte incomparable across rows.


## 1. configuration

In [ ]:
# ---- configuration ----

RUN = "rope"          # ablation row to train.
                      #     "baseline" | "rope" | "rope_gqa" | "rope_gqa_matched"
                      #     Two machines must never use the same value.

DATA_DIR = None       # directory holding the prepared data files.
                      #     None = auto-detect (works on Kaggle if you attached
                      #            the dataset, and locally if files are in ./ )
                      #     Kaggle example: "/kaggle/input/tinystories-bpe8192"
                      #     Windows example: r"C:\Users\you\gpt2data"
                      #     Linux/Mac example: "/home/you/gpt2data"

# ---- fixed across all runs: changing these breaks comparability ----
TARGET_TOKENS = 600_000_000   # locked: baseline used this
INIT_SEED     = 1337          # locked: model init
DATA_SEED     = 4242          # locked: batch order
BATCH_SIZE    = 32            # locked: effective batch (see MICRO_BATCH below)
BLOCK_SIZE    = 512           # locked: context length
EVAL_ITERS    = 100           # locked: size of fixed val set

# --- the ONE performance knob that is safe to change ---------------
MICRO_BATCH   = 32            # reduce on CUDA OOM; must divide BATCH_SIZE.
                              #     Try 16, then 8, then 4. Must divide 32.
                              #     Gradient accumulation keeps the effective
                              #     batch at 32, so results stay identical --
                              #     only speed changes.
TIME_BUDGET_S = 8.0 * 3600    # stop cleanly before a session timeout; safe to edit

## 2. install + imports

In [ ]:
# On Kaggle/Colab this installs the tokenizer lib. On a local machine with
# PyTorch already installed, this is still safe to run.
!pip install -q tokenizers

import os, glob, math, time, json, hashlib
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F

# --- locate data + output folders ---------------------------------
def resolve_paths(data_dir):
    if os.path.isdir("/kaggle/working"):                 # Kaggle
        work = "/kaggle/working"
        if data_dir is None:
            hits = [d for d in glob.glob("/kaggle/input/*") if os.path.exists(f"{d}/train.bin")]
            data_dir = hits[0] if hits else work
    elif os.path.isdir("/content"):                      # Colab
        work = "/content/work"; os.makedirs(work, exist_ok=True)
        if data_dir is None:
            hits = [d for d in ["/content", "/content/drive/MyDrive", work]
                    if os.path.exists(f"{d}/train.bin")]
            data_dir = hits[0] if hits else work
    else:                                                # local PC
        work = os.path.abspath("./work"); os.makedirs(work, exist_ok=True)
        if data_dir is None:
            data_dir = "." if os.path.exists("./train.bin") else work
    return data_dir, work

DATA_DIR, WORK = resolve_paths(DATA_DIR)
print("DATA_DIR :", DATA_DIR, "   <- must contain the 6 files")
print("WORK     :", WORK, "   <- outputs land here")

# --- fail early with a clear message if anything is missing -------
need = ["train.bin","train.bin.meta","val.bin","val.bin.meta",
        "bpe8192-vocab.json","bpe8192-merges.txt"]
missing = [f for f in need if not os.path.exists(f"{DATA_DIR}/{f}")]
assert not missing, f"MISSING in {DATA_DIR}: {missing}\nSet DATA_DIR in Cell 1."

assert torch.cuda.is_available(), "Needs an NVIDIA GPU. CPU/Apple Silicon will not work."
print("\ntorch", torch.__version__, "|", torch.cuda.get_device_name(0),
      "|", f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
dev = "cuda"

## 3. model definition

In [ ]:
"""Decoder-only transformer with switchable positional encoding and KV-head count.

A single Config drives every ablation row:

    row                pos_encoding   n_kv_heads   mlp_ratio
    baseline           learned        8            4.0
    rope               rope           8            4.0
    rope_gqa           rope           2            4.0
    rope_gqa_matched   rope           2            4.75   (param-matched to rope)
"""

import math
from dataclasses import dataclass, asdict

import torch
import torch.nn as nn
import torch.nn.functional as F


def _sdpa_supports_gqa() -> bool:
    """scaled_dot_product_attention is a builtin; probe rather than introspect."""
    try:
        q = torch.zeros(1, 4, 2, 8)
        k = torch.zeros(1, 2, 2, 8)
        F.scaled_dot_product_attention(q, k, k, enable_gqa=True)
        return True
    except Exception:
        return False


SDPA_GQA = _sdpa_supports_gqa()


@dataclass
class Config:
    n_layer: int = 8
    n_head: int = 8
    n_kv_heads: int = 8
    n_embd: int = 512
    block_size: int = 512
    vocab_size: int = 8192
    mlp_ratio: float = 4.0
    dropout: float = 0.0
    pos_encoding: str = "learned"
    rope_theta: float = 10000.0
    max_infer_len: int = 4096

    def __post_init__(self):
        assert self.n_embd % self.n_head == 0
        assert self.n_head % self.n_kv_heads == 0
        assert self.pos_encoding in ("learned", "rope")

    @property
    def head_dim(self) -> int:
        return self.n_embd // self.n_head


def build_rope_cache(head_dim, max_seq, theta, device=None, dtype=torch.float32):
    """Precompute (cos, sin), each of shape (max_seq, head_dim // 2)."""
    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    freqs = torch.outer(torch.arange(max_seq, device=device).float(), inv_freq)
    return freqs.cos().to(dtype), freqs.sin().to(dtype)


def apply_rope(x, cos, sin):
    """Rotate x by the positions encoded in cos/sin.

    x        -- (B, n_head, T, head_dim)
    cos, sin -- (T, head_dim // 2), pre-sliced to absolute positions

    Split-half layout (GPT-NeoX/HF), applied identically to q and k. Accumulated
    in fp32: under fp16 autocast the rotation loses precision at long positions.
    """
    x1, x2 = x.float().chunk(2, dim=-1)
    cos, sin = cos[None, None], sin[None, None]
    out = torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.type_as(x)


class CausalSelfAttention(nn.Module):
    """Multi-head attention; n_kv_heads < n_head selects grouped-query attention."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.n_head = cfg.n_head
        self.n_kv_heads = cfg.n_kv_heads
        self.head_dim = cfg.head_dim
        self.n_rep = cfg.n_head // cfg.n_kv_heads
        self.dropout = cfg.dropout

        self.q_dim = cfg.n_head * cfg.head_dim
        self.kv_dim = cfg.n_kv_heads * cfg.head_dim

        self.qkv = nn.Linear(cfg.n_embd, self.q_dim + 2 * self.kv_dim)
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd)
        self.resid_drop = nn.Dropout(cfg.dropout)

    def forward(self, x, rope=None, kv_cache=None, use_cache=False):
        B, T, C = x.shape
        # Full-sequence prefill or single-token decode only. Chunked prefill against
        # a populated cache would need an explicit mask, since is_causal aligns
        # top-left rather than bottom-right when q_len != k_len.
        assert kv_cache is None or T == 1

        q, k, v = self.qkv(x).split([self.q_dim, self.kv_dim, self.kv_dim], dim=-1)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)

        if rope is not None:
            cos, sin = rope
            q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)  # v unrotated

        if kv_cache is not None:
            past_k, past_v = kv_cache
            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)
        new_cache = (k, v) if use_cache else None

        is_causal = T > 1
        p = self.dropout if self.training else 0.0

        if self.n_rep == 1:
            y = F.scaled_dot_product_attention(q, k, v, is_causal=is_causal, dropout_p=p)
        elif SDPA_GQA:
            y = F.scaled_dot_product_attention(q, k, v, is_causal=is_causal,
                                               dropout_p=p, enable_gqa=True)
        else:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)
            y = F.scaled_dot_product_attention(q, k, v, is_causal=is_causal, dropout_p=p)

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.proj(y)), new_cache


class MLP(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        hidden = int(cfg.mlp_ratio * cfg.n_embd)
        self.fc1 = nn.Linear(cfg.n_embd, hidden)
        self.fc2 = nn.Linear(hidden, cfg.n_embd)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x):
        return self.drop(self.fc2(F.gelu(self.fc1(x))))


class Block(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.n_embd)
        self.mlp = MLP(cfg)

    def forward(self, x, rope=None, kv_cache=None, use_cache=False):
        attn_out, new_cache = self.attn(self.ln1(x), rope, kv_cache, use_cache)
        x = x + attn_out
        x = x + self.mlp(self.ln2(x))
        return x, new_cache


class GPT(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.use_rope = cfg.pos_encoding == "rope"

        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        if self.use_rope:
            cos, sin = build_rope_cache(cfg.head_dim, cfg.max_infer_len, cfg.rope_theta)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
            self.pos_emb = None
        else:
            # Sized to block_size exactly, so the inference ceiling is real and
            # measurable rather than padded around.
            self.pos_emb = nn.Embedding(cfg.block_size, cfg.n_embd)

        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("proj.weight") or name.endswith("fc2.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layer))

    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, kv_caches=None, use_cache=False, start_pos=0):
        B, T = idx.shape
        x = self.tok_emb(idx)

        rope = None
        if self.use_rope:
            # start_pos is the absolute index of idx[:, 0]. During cached decode
            # T == 1 and start_pos advances; slicing from 0 instead trains fine
            # and generates nonsense.
            assert start_pos + T <= self.rope_cos.size(0), "increase max_infer_len"
            rope = (self.rope_cos[start_pos:start_pos + T],
                    self.rope_sin[start_pos:start_pos + T])
        else:
            if start_pos + T > self.pos_emb.num_embeddings:
                raise RuntimeError(
                    f"learned positional table holds {self.pos_emb.num_embeddings} "
                    f"positions, requested {start_pos + T}"
                )
            x = x + self.pos_emb(torch.arange(start_pos, start_pos + T, device=idx.device))

        x = self.drop(x)

        new_caches = []
        for i, block in enumerate(self.blocks):
            x, nc = block(x, rope, kv_caches[i] if kv_caches else None, use_cache)
            new_caches.append(nc)

        x = self.ln_f(x)

        if targets is not None:
            logits = self.head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.reshape(-1))
        else:
            logits = self.head(x[:, [-1], :])  # only the last position is needed
            loss = None

        return logits, loss, (new_caches if use_cache else None)


def param_report(model: GPT, verbose=True):
    """Split the parameter count into embedding and transformer components."""
    total = sum(p.numel() for p in model.parameters())  # tied weights counted once
    emb = model.tok_emb.weight.numel()
    pos = model.pos_emb.weight.numel() if model.pos_emb is not None else 0
    non_emb = total - emb - pos
    if verbose:
        print(f"  total          {total/1e6:7.2f}M")
        print(f"  token emb      {emb/1e6:7.2f}M  ({100*emb/total:.1f}%)")
        print(f"  pos emb        {pos/1e6:7.2f}M")
        print(f"  transformer    {non_emb/1e6:7.2f}M")
    return {"total": total, "emb": emb, "pos": pos, "non_emb": non_emb}


def kv_cache_bytes(cfg: Config, seq_len, batch=1, dtype_bytes=2):
    """2 (K and V) x layers x kv_heads x head_dim x seq_len x batch x bytes."""
    return 2 * cfg.n_layer * cfg.n_kv_heads * cfg.head_dim * seq_len * batch * dtype_bytes


## 4. data + integrity check

**Read the three printed values.** They must match these exactly on every
machine. If any differs, stop — the rows will not be comparable.

```
tokenizer sha256[:16] : 87302aa2fe41fad2
train tokens          : 466752940
first-batch checksum  : (whatever machine 1 prints -- record it)
```

In [ ]:
meta_t = json.load(open(f"{DATA_DIR}/train.bin.meta"))
meta_v = json.load(open(f"{DATA_DIR}/val.bin.meta"))
VOCAB_SIZE = meta_t.get("vocab_size", 8192)
BPB_RATIO  = meta_v["n_tokens"] / meta_v["n_bytes"]   # nats->bpb conversion

train_arr = np.memmap(f"{DATA_DIR}/train.bin", dtype=np.uint16, mode="r")
val_arr   = np.memmap(f"{DATA_DIR}/val.bin",   dtype=np.uint16, mode="r")

h = hashlib.sha256()
for f in ["bpe8192-vocab.json", "bpe8192-merges.txt"]:
    h.update(open(f"{DATA_DIR}/{f}", "rb").read())
print("tokenizer sha256[:16] :", h.hexdigest()[:16], "  (expect 87302aa2fe41fad2)")
print("train tokens          :", train_arr.size, "  (expect 466752940)")
print("val tokens            :", val_arr.size,   "  (expect 4691115)")
print("tokens/byte           :", round(BPB_RATIO, 5))

# --- batching -----------------------------------------------------
# Draws BATCH_SIZE indices from a DEDICATED generator. This is seeded
# separately from model init, so every config sees the SAME batch order.
# (Using the global RNG here was the bug in the earlier notebook: each
#  config consumes a different amount of RNG during init, so each run
#  silently trained on a different data stream.)
def _draw(arr, gen):
    ix = torch.randint(len(arr) - BLOCK_SIZE - 1, (BATCH_SIZE,), generator=gen)
    x = torch.stack([torch.from_numpy(arr[i:i+BLOCK_SIZE].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(arr[i+1:i+1+BLOCK_SIZE].astype(np.int64)) for i in ix])
    return x.pin_memory().to(dev, non_blocking=True), y.pin_memory().to(dev, non_blocking=True)

_vg = torch.Generator().manual_seed(12345)            # fixed val set, never changes
VAL_BATCHES = [_draw(val_arr, _vg) for _ in range(EVAL_ITERS)]
print(f"\nfixed val set: {EVAL_ITERS} batches x {BATCH_SIZE} x {BLOCK_SIZE}")

## 5. build the model

In [ ]:
RUNS = {                                    # pos_encoding, kv heads, mlp width
  "baseline":         dict(pos_encoding="learned", n_kv_heads=8, mlp_ratio=4.0),
  "rope":             dict(pos_encoding="rope",    n_kv_heads=8, mlp_ratio=4.0),
  "rope_gqa":         dict(pos_encoding="rope",    n_kv_heads=2, mlp_ratio=4.0),
  "rope_gqa_matched": dict(pos_encoding="rope",    n_kv_heads=2, mlp_ratio=4.75),
}
assert RUN in RUNS, f"RUN must be one of {list(RUNS)}"
assert BATCH_SIZE % MICRO_BATCH == 0, "MICRO_BATCH must divide BATCH_SIZE"
ACCUM = BATCH_SIZE // MICRO_BATCH

torch.manual_seed(INIT_SEED); np.random.seed(INIT_SEED)
cfg = Config(n_layer=8, n_head=8, n_embd=512, block_size=BLOCK_SIZE,
             vocab_size=VOCAB_SIZE, dropout=0.0, max_infer_len=4096, **RUNS[RUN])
model = GPT(cfg).to(dev)
print(f"[{RUN}]  pos={cfg.pos_encoding}  n_kv_heads={cfg.n_kv_heads}  mlp={cfg.mlp_ratio}")
rep = param_report(model)

# Seeded AFTER model init on purpose -- see note in Cell 4.
data_gen = torch.Generator().manual_seed(DATA_SEED)
def get_batch(): return _draw(train_arr, data_gen)

_p = torch.Generator().manual_seed(DATA_SEED)
print("\nfirst-batch checksum :", int(torch.randint(
      len(train_arr)-BLOCK_SIZE-1, (BATCH_SIZE,), generator=_p).sum()),
      " <- MUST match on every machine")

MAX_ITERS = TARGET_TOKENS // (BATCH_SIZE * BLOCK_SIZE)
print(f"max_iters {MAX_ITERS}  |  micro_batch {MICRO_BATCH} x accum {ACCUM}")
print(f"tokens/param {TARGET_TOKENS/rep['total']:.1f} (Chinchilla ~20)")

## 6. optimizer + eval

In [ ]:
LR, MIN_LR_FRAC, WARMUP, WD, GRAD_CLIP, EVAL_INTERVAL = 6e-4, 0.1, 500, 0.1, 1.0, 1000

def get_lr(it):
    if it < WARMUP: return LR * (it + 1) / WARMUP
    r = min((it - WARMUP) / max(1, MAX_ITERS - WARMUP), 1.0)
    return MIN_LR_FRAC*LR + 0.5*(1+math.cos(math.pi*r))*(1-MIN_LR_FRAC)*LR

# no weight decay on 1-D params (LayerNorm gains, biases) -- GPT-2 practice
decay   = [p for _, p in model.named_parameters() if p.dim() >= 2]
nodecay = [p for _, p in model.named_parameters() if p.dim() <  2]
optimizer = torch.optim.AdamW([{"params": decay, "weight_decay": WD},
                               {"params": nodecay, "weight_decay": 0.0}],
                              lr=LR, betas=(0.9, 0.95), fused=True)
scaler = torch.amp.GradScaler("cuda")

@torch.no_grad()
def evaluate():
    model.eval(); tot = 0.0
    for xb, yb in VAL_BATCHES:
        with torch.amp.autocast("cuda", dtype=torch.float16):
            _, loss, _ = model(xb, yb)
        tot += loss.item()
    model.train()
    nll = tot / len(VAL_BATCHES)
    return nll, (nll / math.log(2)) * BPB_RATIO      # bits-per-byte

## 7. TRAIN (~2.6 h on a T4)

Safe to interrupt. Re-running this cell resumes from the last checkpoint,
including the batch-order generator state — so it does not replay data it has
already seen.

In [ ]:
CKPT = f"{WORK}/ckpt_{RUN}.pt"
start_iter, best_bpb, log = 0, float("inf"), []

if os.path.exists(CKPT):                      # auto-resume
    sd = torch.load(CKPT, map_location=dev)
    model.load_state_dict(sd["model"]);   optimizer.load_state_dict(sd["optim"])
    scaler.load_state_dict(sd["scaler"]); data_gen.set_state(sd["data_gen"].cpu())
    start_iter, best_bpb, log = sd["iter"]+1, sd["best_bpb"], sd["log"]
    print(f"RESUMED at iter {start_iter}, best bpb {best_bpb:.4f}")

model.train(); torch.cuda.synchronize(); t0 = time.time()

for it in range(start_iter, MAX_ITERS):
    for g in optimizer.param_groups: g["lr"] = get_lr(it)

    xb, yb = get_batch()                      # full batch of 32, drawn once
    optimizer.zero_grad(set_to_none=True)
    for k in range(ACCUM):                    # split into micro-batches if needed
        s = slice(k*MICRO_BATCH, (k+1)*MICRO_BATCH)
        with torch.amp.autocast("cuda", dtype=torch.float16):
            _, loss, _ = model(xb[s], yb[s])
            loss = loss / ACCUM               # mean over the full batch
        scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    scaler.step(optimizer); scaler.update()

    if it % EVAL_INTERVAL == 0 or it == MAX_ITERS - 1:
        nll, bpb = evaluate(); el = time.time() - t0
        log.append({"iter": it, "val_nll": nll, "val_bpb": bpb, "elapsed": el})
        eta = (MAX_ITERS - it) / max(1, it - start_iter + 1) * el / 3600
        print(f"iter {it:6d} | nll {nll:.4f} | bpb {bpb:.4f} | {el/60:6.1f}m | eta {eta:.1f}h")
        if bpb < best_bpb:
            best_bpb = bpb
            torch.save({"model": model.state_dict(), "cfg": asdict(cfg)}, f"{WORK}/best_{RUN}.pt")
        torch.save({"model": model.state_dict(), "optim": optimizer.state_dict(),
                    "scaler": scaler.state_dict(), "data_gen": data_gen.get_state(),
                    "iter": it, "best_bpb": best_bpb, "log": log}, CKPT)
        if el > TIME_BUDGET_S:
            print("time budget hit -- just re-run this cell to resume"); break

torch.cuda.synchronize()
print(f"\nDONE. best val bpb {best_bpb:.4f}  ({(time.time()-t0)/60:.1f} min)")

## 8. benchmarks + write results

`baseline` will error at ctx 1024/2048 — its position table stops at 512. That
error is a result, not a failure. The RoPE rows should run all three.

In [ ]:
model.load_state_dict(torch.load(f"{WORK}/best_{RUN}.pt", map_location=dev)["model"]); model.eval()

@torch.no_grad()
def gen_cached(m, idx, n_new, temp=0.8):
    lg, _, kc = m(idx, use_cache=True, start_pos=0); pos = idx.shape[1]
    for _ in range(n_new):
        nxt = torch.multinomial(F.softmax(lg[:, -1, :]/temp, -1), 1)
        idx = torch.cat([idx, nxt], 1)
        lg, _, kc = m(nxt, use_cache=True, kv_caches=kc, start_pos=pos); pos += 1
    return idx

def timeit(fn, n=3):
    fn(); torch.cuda.synchronize()
    ts = []
    for _ in range(n):
        torch.cuda.synchronize(); t = time.time(); fn(); torch.cuda.synchronize()
        ts.append(time.time()-t)
    return min(ts)

results = {"run": RUN, "params": rep["total"], "non_emb": rep["non_emb"],
           "best_bpb": best_bpb, "n_kv_heads": cfg.n_kv_heads,
           "mlp_ratio": cfg.mlp_ratio, "pos": cfg.pos_encoding,
           "gpu": torch.cuda.get_device_name(0), "log": log, "ctx": {}}

for ctx in (256, 1024, 2048):
    idx = torch.randint(0, VOCAB_SIZE, (1, ctx//2), device=dev)
    row = {"kv_cache_MB": kv_cache_bytes(cfg, ctx)/1e6}
    try:
        row["prefill_s"] = timeit(lambda: model(idx, use_cache=True, start_pos=0))
        t = timeit(lambda: gen_cached(model, idx, ctx//2))
        row["decode_tok_s"] = (ctx//2)/(t - row["prefill_s"])
    except RuntimeError as e:
        row["error"] = str(e)[:120]
    results["ctx"][ctx] = row
    print(f"ctx {ctx:5d} | cache {row['kv_cache_MB']:6.2f} MB | " +
          (row.get("error") or
           f"prefill {row['prefill_s']*1e3:6.1f} ms | decode {row['decode_tok_s']:7.1f} tok/s"))

json.dump(results, open(f"{WORK}/results_{RUN}.json", "w"), indent=2)

from tokenizers import ByteLevelBPETokenizer
tk = ByteLevelBPETokenizer(f"{DATA_DIR}/bpe8192-vocab.json", f"{DATA_DIR}/bpe8192-merges.txt")
samples = []
for p in ["Once upon a time", "The little girl saw a", "Tom and his dog"]:
    ids = torch.tensor([tk.encode(p).ids], device=dev)
    samples.append(tk.decode(gen_cached(model, ids, 150)[0].tolist()))
open(f"{WORK}/sample_{RUN}.txt","w").write("\n\n---\n\n".join(samples))
print("\n" + samples[0][:400])

## 9. zip and send back

**Send the zip only** (a few hundred KB). Keep the `.pt` files on your own
machine — they are ~120 MB and ~350 MB and are not needed for the comparison.

In [ ]:
import zipfile
zp = f"{WORK}/ablation_{RUN}.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in [f"results_{RUN}.json", f"sample_{RUN}.txt"]:
        p = f"{WORK}/{f}"
        if os.path.exists(p): z.write(p, f); print(" +", f)
        else: print(" - MISSING", f)
print(f"\n{zp}  ({os.path.getsize(zp)/1e3:.0f} KB)")

try:
    from IPython.display import FileLink, display
    display(FileLink(zp))          # local / Colab download link
except Exception:
    pass                           # on Kaggle: right sidebar -> Output -> click zip